In [38]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader(
    "User-Agent",
    "MyResearchProject/1.0 (your_email@example.com)"
)

query = """
SELECT ?company ?companyLabel ?companyId
       (SAMPLE(?ticker0) AS ?ticker)
       (SAMPLE(?industryLabel0) AS ?industry)
WHERE {
  ?company wdt:P414 wd:Q13677 .

  OPTIONAL {
    ?company p:P414 ?listingStatement .
    ?listingStatement ps:P414 wd:Q13677 .
    ?listingStatement pq:P249 ?ticker0 .
  }

  OPTIONAL {
    ?company wdt:P452 ?industry0 .
    ?industry0 rdfs:label ?industryLabel0 .
    FILTER(LANG(?industryLabel0) = "en")
  }

  BIND(REPLACE(STR(?company), "http://www.wikidata.org/entity/", "") AS ?companyId)

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en".
  }
}
GROUP BY ?company ?companyLabel ?companyId
ORDER BY ?companyLabel
LIMIT 5000
"""

sparql.setQuery(query)
results = sparql.query().convert()

rows = []
for row in results["results"]["bindings"]:
    rows.append({
        "company": row.get("companyLabel", {}).get("value"),
        "company_id": row.get("companyId", {}).get("value"),
        "ticker": row.get("ticker", {}).get("value"),
        "industry": row.get("industry", {}).get("value"),
        "company_url": row.get("company", {}).get("value"),
    })

df = pd.DataFrame(rows, columns=["company", "company_id", "ticker", "industry", "company_url"])

df.to_csv("nyse_companies.csv", index=False, encoding="utf-8")

print("Saved to nyse_companies.csv")
print(df.head(20))
print("Total companies:", len(df))

Saved to nyse_companies.csv
                 company  company_id ticker                  industry  \
0             3D Systems    Q4636301    DDD               3D printing   
1    4Kids Entertainment     Q604775    KDE                       NaN   
2                 58.com   Q17499083   WUBA                       NaN   
3             7 Days Inn    Q4643844    NaN                     hotel   
4                    8x8    Q4645515   EGHT                       NaN   
5   99 Cents Only Stores    Q4646294    NDN            discount store   
6           A10 Networks   Q16157155   ATEN          computer network   
7               AAR Corp    Q4649932    AIR        aerospace industry   
8              ABB Group      Q52825    ABB    electrical engineering   
9         ABM Industries    Q4650338    ABM       facility management   
10           ACCO Brands     Q288129   ACCO  industrial manufacturing   
11              ADT Inc.     Q290680    ADT                       NaN   
12                 AECO

In [39]:
import pandas as pd
import yfinance as yf

target_date = pd.Timestamp("2018-01-01")

def market_cap_on_or_before_date(ticker):
    if pd.isna(ticker) or str(ticker).strip() == "":
        return None

    try:
        t = yf.Ticker(str(ticker).strip())

        price_hist = t.history(
            start=(target_date - pd.Timedelta(days=10)).strftime("%Y-%m-%d"),
            end=(target_date + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
            auto_adjust=False
        )
        if price_hist.empty:
            return None

        price_hist.index = pd.to_datetime(price_hist.index).tz_localize(None)
        price_hist = price_hist[price_hist.index <= target_date]
        if price_hist.empty:
            return None

        close_price = float(price_hist.iloc[-1]["Close"])

        shares = t.get_shares_full(
            start=(target_date - pd.Timedelta(days=370)).strftime("%Y-%m-%d"),
            end=(target_date + pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
        )
        if shares is None or len(shares) == 0:
            return None

        shares.index = pd.to_datetime(shares.index).tz_localize(None)
        shares = shares[shares.index <= target_date]
        if shares.empty:
            return None

        shares_outstanding = float(shares.iloc[-1])

        return close_price * shares_outstanding

    except Exception:
        return None

df["market_cap_2026_01_01"] = df["ticker"].apply(market_cap_on_or_before_date)

$KDE: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03)
$WUBA: possibly delisted; no timezone found
$NDN: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03)
$ABB: possibly delisted; no timezone found
$ADT: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03) (Yahoo error = "Data doesn't exist for startDate = 1513918800, endDate = 1514955600")
$GAS: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03)
$HKD: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03) (Yahoo error = "Data doesn't exist for startDate = 1513918800, endDate = 1514955600")
$AUO: possibly delisted; no timezone found
$AVG: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-01-03) (Yahoo error = "Data doesn't exist for startDate = 1513918800, endDate = 1514955600")
$AAN: possibly delisted; no timezone found
$ACCD: possibly delisted; no timezone found
$ACT: possibly delisted; no price data found  (1d 2017-12-22 -> 2018-

In [40]:
df.to_csv("nyse_companies_cap.csv", index=False, encoding="utf-8")


In [41]:
df_with_market_cap = df[df["market_cap_2026_01_01"].notna()].copy()

print("Number of companies with market cap:", len(df_with_market_cap))

df_with_market_cap = df_with_market_cap.sort_values(
    "market_cap_2026_01_01",
    ascending=False
).reset_index(drop=True)

print(df_with_market_cap.head(20))

Number of companies with market cap: 1056
                   company company_id ticker                  industry  \
0                  Clarcor  Q16958713    CLC               gas turbine   
1             GE Aerospace   Q1485061     GE        aerospace industry   
2         General Electric     Q54173     GE    mechanical engineering   
3            Alibaba Group   Q1359568   BABA                e-commerce   
4           JPMorgan Chase    Q192314    JPM        financial services   
5        Johnson & Johnson    Q333718    JNJ   pharmaceutical industry   
6               ExxonMobil    Q156238    XOM        petroleum industry   
7          Bank of America    Q487907    BAC        financial services   
8              Wells Fargo    Q744149    WFC        financial services   
9                     Visa    Q328840      V        financial services   
10     Chevron Corporation    Q319642    CVX        petroleum industry   
11        Procter & Gamble    Q212405     PG   pharmaceutical industry

In [42]:
df_with_market_cap.to_csv("nyse_companies_with_market_cap_sorted.csv", index=False, encoding="utf-8")
print("Saved to nyse_companies_with_market_cap_sorted.csv")

Saved to nyse_companies_with_market_cap_sorted.csv
